# 🧠 Brain Tumor Project — Full GPU Training (Colab)

Yeh notebook poora post-fix pipeline chalata hai: BRISC training data par **U-Net + Experiment 1/2/3** training, Grad-CAM localization, PMRAM external validation, aur consolidated reports.

**Pehle: Runtime → Change runtime type → GPU (T4) select karo.** Phir cells top-to-bottom run karo.

Agar Colab disconnect ho jaye: dobara connect karke **sirf Step 7** dobara chalao — pipeline `--from-stage` se wahin se resume karega jahan ruka tha.

## Step 1 — GPU check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime type me GPU select karo!")

## Step 2 — Drive mount (checkpoints/reports save rahenge)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BACKUP_DIR = "/content/drive/MyDrive/brain_tumor_project"
os.makedirs(BACKUP_DIR, exist_ok=True)
print("Backup dir:", BACKUP_DIR)

## Step 3 — Repo clone + dependencies

In [ ]:
%cd /content
import os
if not os.path.exists("/content/Brain_Tumor_Project"):
    !git clone https://github.com/CSwithUnkit/Brain_Tumor_Project.git
else:
    %cd /content/Brain_Tumor_Project
    !git pull
    %cd /content
%cd /content/Brain_Tumor_Project
!pip install -q -r requirements.txt
print("deps done")

## Step 4 — BRISC dataset download (Zenodo, ~260 MB, no login)

In [ ]:
%cd /content/Brain_Tumor_Project
!mkdir -p data/brisc
%cd data/brisc
!curl -sL --retry 5 --retry-all-errors -C - -o brisc2025.zip "https://zenodo.org/records/17524350/files/brisc2025.zip?download=1"
!unzip -n -q brisc2025.zip
!find brisc2025/classification_task -mindepth 2 -maxdepth 2 -type d | while read d; do echo "$d: $(ls "$d" | wc -l)"; done

## Step 5 — PMRAM raw/original images (Mendeley, ~38 MB, no login)

External validation ke liye **sirf original 1600 images** use hoti hain (augmented nahi — leakage se bachne ke liye).

In [ ]:
%cd /content/Brain_Tumor_Project
!mkdir -p data/pmram
%cd data/pmram
!curl -sL --retry 5 --retry-all-errors -C - -o raw_data.zip "https://data.mendeley.com/public-files/datasets/m7w55sw88b/files/92239c50-c554-48a5-8e6e-7d01a22ede01/file_downloaded"
!unzip -n -q raw_data.zip
# 'Raw/' folder seedha use hota hai — validation ka folder-walk mode
# 512Glioma/512Meningioma/512Pituitary/512Normal ko classes me map karta hai;
# 'augmented' naam ki koi cheez yahan hai hi nahi, to rename ki zaroorat nahi
!ls Raw/ && find Raw -type f | wc -l


## Step 6 — Ingestion audit (counts verify karo)

In [ ]:
%cd /content/Brain_Tumor_Project
!python -m data.dataset_ingestion 2>&1 | tail -5
!python -c "
import json
m = json.load(open('data/brisc/brisc_metadata.json'))
print('classification:', m['classification_count'], '| segmentation:', m['segmentation_count'])
"

## Step 7 — Full pipeline (cache → U-Net → Exp1/2/3 → XAI → PMRAM → reports)

⏳ **Yeh sabse lamba step hai (T4 par kai ghante).** Browser tab khula rakho.

Agar beech me ruke: dobara connect karke yeh cell chalao — `--from-stage` + `--skip-cache` se resume hoga:
`!python scripts/rerun_full_pipeline.py --from-stage <ruka_hua_stage> --skip-cache`
(stages: cache, unet, exp1, exp2, exp3, xai, pmram, reports)

In [ ]:
%cd /content/Brain_Tumor_Project
!python scripts/rerun_full_pipeline.py 2>&1 | tee /content/drive/MyDrive/brain_tumor_project/pipeline_run.log

## Step 8 — Checkpoints + reports Drive me backup karo

In [ ]:
%cd /content/Brain_Tumor_Project
!cp -r checkpoints "$BACKUP_DIR/" 2>/dev/null; cp -r results "$BACKUP_DIR/" 2>/dev/null; cp -r reports "$BACKUP_DIR/" 2>/dev/null
!ls -la "$BACKUP_DIR" && du -sh "$BACKUP_DIR"/*
print("\n✅ Backup complete. checkpoints/ aur results/ ab Drive me safe hain.")